# Module 12: Black-box optimisation — initial data analysis

This notebook examines the eight supplied datasets, visualises observations, and illustrates Gaussian Process predictions. **The true functions are unknown; GP curves are estimates, not ground truth.**

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
DATA = Path("../data/initial_data")

In [ ]:
datasets = {}
for i in range(1, 9):
    folder = DATA / f"function_{i}"
    X = np.load(folder / "initial_inputs.npy")
    y = np.load(folder / "initial_outputs.npy").ravel()
    datasets[i] = (X, y)
    print(f"Function {i}: X={X.shape}, y={y.shape}, best observed={y.max():.6f}")

## Plot observed inputs for the 2D functions

In [ ]:
for i in (1, 2):
    X, y = datasets[i]
    fig, ax = plt.subplots(figsize=(6, 5))
    points = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=80, edgecolor="black")
    fig.colorbar(points, ax=ax, label="Observed output")
    ax.set(xlabel="Input 1", ylabel="Input 2", title=f"Function {i}: initial data")
    plt.show()

## Plot observed outputs for all eight functions

In [ ]:
for i, (X, y) in datasets.items():
    plt.figure(figsize=(7, 3))
    plt.scatter(np.arange(1, len(y)+1), y)
    plt.axhline(y.max(), linestyle="--", color="gray")
    plt.title(f"Function {i}: observed outputs")
    plt.xlabel("Observation number")
    plt.ylabel("Output")
    plt.show()

## Gaussian Process slice (illustrative)

For each function, fit a GP to the supplied data and vary the first input while holding other inputs at their best observed values. This is a **one-dimensional slice**, not a full plot of the unknown function.

In [ ]:
for i, (X, y) in datasets.items():
    d = X.shape[1]
    kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(d), nu=2.5) + WhiteKernel(noise_level=1e-4)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=2, random_state=42)
    gp.fit(X, y)
    x_best = X[np.argmax(y)].copy()
    grid = np.linspace(0, 1, 200)
    slice_X = np.tile(x_best, (len(grid), 1))
    slice_X[:, 0] = grid
    mean, std = gp.predict(slice_X, return_std=True)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(grid, mean, label="GP predicted mean")
    ax.fill_between(grid, mean-1.96*std, mean+1.96*std, alpha=0.2, label="Approx. 95% interval")
    ax.axvline(x_best[0], color="gray", linestyle="--", label="Best observed input 1")
    ax.set(xlabel="Input 1 (other inputs fixed)", ylabel="Predicted output", title=f"Function {i}: GP slice")
    ax.legend()
    fig.savefig(Path("../images") / f"function_{i}_gp_slice.png", dpi=150, bbox_inches="tight")
    plt.show()

## Reflection notes
- Why did I select each query?
- Which functions had greater uncertainty or noise?
- How will I revise my approach after receiving the next outputs?

**Do not label the model estimates as true maxima.**